# TP  Knowledge Distillation for image classification

The goal of this lab is to practice the concept of knowledge distillation between neural networks
for image classification. You will code in Python using the PyTorch library on Google Colab.

## Using existing Convolutional Neural Networks

In this section, the teacher network will be ResNet50, and the student network will be ResNet18.
The MSE (Mean Squared Error) metric will be used to compare these two networks

### Teacher Training
1. Train the teacher network (pre-trained on ImageNet) on the CIFAR10 dataset while evaluating it.
Ensure proper data split for training, validation, and testing. Calculate the number of parameters

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

#store chkpt
save_dir = '/content/drive/MyDrive/saved_models'
os.makedirs(save_dir, exist_ok=True)
best_model_path = os.path.join(save_dir, 'best_resnet50_teacher.pth')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torchvision
from torchvision import models
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import time
from torchsummary import summary
import numpy as np

In [3]:
BATCH_SIZE = 64

# data
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
]) #resize to expected input

full_train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)

val_size = int(0.1 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

100%|██████████| 170M/170M [00:16<00:00, 10.3MB/s]


In [4]:
NUM_EPOCHS = 10

#model
model_teacher = models.resnet50(pretrained=True)
#cifar10 =10classes
model_teacher.fc = torch.nn.Linear(model_teacher.fc.in_features, 10)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_teacher.to(device)

#num params +size model
summary(model_teacher, (3, 224, 224), device=device.type)

#training
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_teacher.parameters(), lr=0.001)
best_val_acc = 0.0
start_time = time.time()
for epoch in range(NUM_EPOCHS):
    #train
    model_teacher.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_teacher(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    #val
    model_teacher.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_teacher(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Train Loss: {epoch_loss:.4f} - Val Loss: {epoch_val_loss:.4f} - Val Acc: {val_accuracy:.2f}%")
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model_teacher.state_dict(), best_model_path)

end_time = time.time()
temps_entrainement = end_time - start_time
print(f"Training time: {temps_entrainement:.2f} seconds")

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 181MB/s]


Using device: cuda
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
           Conv2d-11          [-1, 256, 56, 56]          16,384
      BatchNorm2d-12          [-1, 256, 56, 56]             512
           Conv2d-13          [-1, 256, 56, 56]          16,384
      BatchNorm2d-14

In [5]:
# test

model_teacher.load_state_dict(torch.load(best_model_path))
model_teacher.eval()
test_loss = 0.0
correct = 0
total = 0
start = time.time()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_teacher(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
end = time.time()
average_test_loss = test_loss / len(test_loader)
test_accuracy = 100 * correct / total

temps_total = end - start
nb_images = len(test_dataset)
fps = nb_images / temps_total
print(f"Teacher Model Test Evaluation:")
print(f"Average Test Loss: {average_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Teacher Model Test Evaluation:
Average Test Loss: 0.3625
Test Accuracy: 88.69%


In [6]:
from torch.utils.flop_counter import FlopCounterMode

sample_images, _ = next(iter(test_loader))
dummy_input = sample_images[:1].to(device)

model_teacher.load_state_dict(torch.load(best_model_path))
model_teacher.eval()
with torch.no_grad():
    with FlopCounterMode(display=False) as flop_counter:
        model_teacher(dummy_input)

flops = flop_counter.get_total_flops()
macs = flops / 2

print(f"FLOPs: {flops / 1e9:.3f} GFLOPs")
print(f"MACs:  {macs / 1e9:.3f} GMACs")

FLOPs: 8.174 GFLOPs
MACs:  4.087 GMACs


### Strategy 1: Knowledge distillation using a pre-trained Student
2. Train the student network (pre-trained on ImageNet) on the CIFAR10 dataset while evaluating it.
Ensure proper data split for training, validation, and testing. Calculate the number of parameters



In [9]:
#student network training
model_student=models.resnet18(pretrained=True)
best_student1_path= os.path.join(save_dir, 'best_student_Q1.pth')
model_student.fc = torch.nn.Linear(model_student.fc.in_features, 10)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_student.to(device)

#num params +size model
summary(model_student, (3, 224, 224), device=device.type)
best_val_acc = 0.0

#training
start_time = time.time()
for epoch in range(NUM_EPOCHS):
    #train
    model_student.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_student(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    #val
    model_student.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_student(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model_student.state_dict(), best_student1_path)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Train Loss: {epoch_loss:.4f} - Val Loss: {epoch_val_loss:.4f} - Val Acc: {val_accuracy:.2f}%")

end_time = time.time()
temps_entrainement = end_time - start_time
print(f"Training time: {temps_entrainement:.2f} seconds")

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Using device: cuda
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]          36,864
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
       BasicBlock-11           [-1, 64, 56, 56]               0
           Conv2d-12           [-1, 64, 56, 56]          36,864
      BatchNorm2d-13           [-1, 64, 56, 56]             128
             ReLU-14

In [10]:
# test
model_student.load_state_dict(torch.load(best_student1_path))
model_student.eval()
test_loss = 0.0
correct = 0
total = 0
start = time.time()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_student(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
end = time.time()
average_test_loss = test_loss / len(test_loader)
test_accuracy = 100 * correct / total

temps_total = end - start
nb_images = len(test_dataset)
fps = nb_images / temps_total
print(f"Student Model Test Evaluation:")
print(f"Average Test Loss: {average_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Student Model Test Evaluation:
Average Test Loss: 2.6106
Test Accuracy: 9.99%


In [11]:
from torch.utils.flop_counter import FlopCounterMode

sample_images, _ = next(iter(test_loader))
dummy_input = sample_images[:1].to(device)

model_student.eval()
with torch.no_grad():
    with FlopCounterMode(display=False) as flop_counter:
        model_student(dummy_input)

flops = flop_counter.get_total_flops()
macs = flops / 2

print(f"FLOPs: {flops / 1e9:.3f} GFLOPs")
print(f"MACs:  {macs / 1e9:.3f} GMACs")

FLOPs: 3.627 GFLOPs
MACs:  1.814 GMACs


## knowledge distillation student training method


In [22]:
import torch.nn.functional as F
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
alpha=0.5
beta=0.5
gamma=0.25
temperature=4.0

def compute_distance(student_out, teacher_out, metric="mse", temperature=4.0):
    """Computes the distillation distance between student and teacher tensors."""
    if metric == "mse":
        return F.mse_loss(student_out, teacher_out)
    elif metric == "kl":
        # Standard KL Divergence for logits (scores)
        return F.kl_div(
            F.log_softmax(student_out / temperature, dim=1),
            F.softmax(teacher_out / temperature, dim=1),
            reduction="batchmean",
        ) * (temperature ** 2)
    elif metric == "cosine":
        # Flattens spatial dims if feature maps are passed
        s_flat = student_out.flatten(1)
        t_flat = teacher_out.flatten(1)
        return (1.0 - F.cosine_similarity(s_flat, t_flat, dim=1)).mean()
    else:
        raise ValueError(f"Unsupported metric: {metric}")


def train_student(save_path, pretrained=True, use_features=False, logits_metric="mse", feat_metric="mse"):
    #student
    student = models.resnet18(pretrained=pretrained)
    student.fc = nn.Linear(student.fc.in_features, 10)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    student = student.to(device)

    #teacher
    model_teacher.eval()
    for param in model_teacher.parameters():
        param.requires_grad = False


    teacher_feats, student_feats = {}, {}
    handles = []
    params_to_optimize = list(student.parameters())
    adapter = None

    if use_features:
        def hook_fn(storage, name):
            def hook(module, inp, out):
                storage[name] = out
            return hook

        handles.append(model_teacher.layer4.register_forward_hook(hook_fn(teacher_feats, "layer4")))
        handles.append(student.layer4.register_forward_hook(hook_fn(student_feats, "layer4")))

        adapter = nn.Conv2d(512, 2048, kernel_size=1).to(device)
        params_to_optimize += list(adapter.parameters())

    ce_criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(params_to_optimize, lr=LEARNING_RATE)

    best_val_acc = 0.0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    start_time = time.time()

    try:
        for epoch in range(NUM_EPOCHS):
            #train
            student.train()
            if adapter is not None:
                adapter.train()
            model_teacher.eval()

            running_loss = 0.0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.no_grad():
                    teacher_logits = model_teacher(images)

                student_logits = student(images)

                #loss
                loss_ce = ce_criterion(student_logits, labels)
                loss_logits = compute_distance(student_logits, teacher_logits, logits_metric, temperature)
                loss = alpha * loss_ce + beta * loss_logits

                #si fmap utilisées
                if use_features:
                    s_feat = adapter(student_feats["layer4"])
                    t_feat = teacher_feats["layer4"]
                    loss_feat = compute_distance(s_feat, t_feat, feat_metric)
                    loss = loss + gamma * loss_feat

                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            epoch_train_loss = running_loss / len(train_loader)

            #val
            student.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = student(images)
                    val_loss += ce_criterion(outputs, labels).item()

                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels).sum().item()

            epoch_val_loss = val_loss / len(val_loader)
            val_acc = 100.0 * correct / total
            end_time = time.time()
            history["train_loss"].append(epoch_train_loss)
            history["val_loss"].append(epoch_val_loss)
            history["val_acc"].append(val_acc)

            print(
                f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
                f"- Train Loss: {epoch_train_loss:.4f} "
                f"- Val Loss: {epoch_val_loss:.4f} "
                f"- Val Acc: {val_acc:.2f}%"
            )

            #save best
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(student.state_dict(), save_path)
                print(f"  -> Saved new best student ({best_val_acc:.2f}%) to {save_path}")

    finally:
        # remove hook to avoid memory leaks
        for h in handles:
            h.remove()
    temps_entrainement = end_time - start_time
    print(f"Best Val Acc: {best_val_acc:.2f}%\n")

    student.load_state_dict(torch.load(save_path, map_location=device))
    return student, history, temps_entrainement

3. Freeze the teacher network's parameters and train the student network under the supervision of
the teacher using knowledge distillation at the scores level

In [23]:
studentQ3, hist3, time3 = train_student(save_path=os.path.join(save_dir, "student_Q3.pth"),pretrained=True, use_features=False,logits_metric="mse",feat_metric="mse")
print(f"Temps d'entraînement Student (pre-trained, scores only) : {time3:.2f} ")

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/10] - Train Loss: 2.8519 - Val Loss: 0.3451 - Val Acc: 88.50%
  -> Saved new best student (88.50%) to /content/drive/MyDrive/saved_models/student_Q3.pth
Epoch [2/10] - Train Loss: 1.5129 - Val Loss: 0.3669 - Val Acc: 88.20%
Epoch [3/10] - Train Loss: 1.2247 - Val Loss: 0.3004 - Val Acc: 90.06%
  -> Saved new best student (90.06%) to /content/drive/MyDrive/saved_models/student_Q3.pth
Epoch [4/10] - Train Loss: 1.0203 - Val Loss: 0.3230 - Val Acc: 89.48%
Epoch [5/10] - Train Loss: 0.9080 - Val Loss: 0.3328 - Val Acc: 89.42%
Epoch [6/10] - Train Loss: 0.8080 - Val Loss: 0.3066 - Val Acc: 89.72%
Epoch [7/10] - Train Loss: 0.7617 - Val Loss: 0.3087 - Val Acc: 89.62%
Epoch [8/10] - Train Loss: 0.7057 - Val Loss: 0.3456 - Val Acc: 88.72%
Epoch [9/10] - Train Loss: 0.6526 - Val Loss: 0.3069 - Val Acc: 89.80%
Epoch [10/10] - Train Loss: 0.6265 - Val Loss: 0.3102 - Val Acc: 89.46%
Best Val Acc: 90.06%

Temps d'entraînement Student (pre-trained, scores only) : 1903.29 


4. Freeze the teacher network's parameters and train the student network under the supervision of
the teacher using knowledge distillation at both the scores and feature map levels

In [25]:
studentQ4, hist4, time4 = train_student(save_path=os.path.join(save_dir, "student_Q4.pth"),pretrained=True, use_features=True,logits_metric="mse",feat_metric="mse")
print(f"Temps d'entraînement Student (pre-trained, scores and features) : {time4:.2f} ")

Epoch [1/10] - Train Loss: 2.8776 - Val Loss: 0.3594 - Val Acc: 88.48%
  -> Saved new best student (88.48%) to /content/drive/MyDrive/saved_models/student_Q4.pth
Epoch [2/10] - Train Loss: 1.5226 - Val Loss: 0.2830 - Val Acc: 90.62%
  -> Saved new best student (90.62%) to /content/drive/MyDrive/saved_models/student_Q4.pth
Epoch [3/10] - Train Loss: 1.2093 - Val Loss: 0.2988 - Val Acc: 90.12%
Epoch [4/10] - Train Loss: 1.0430 - Val Loss: 0.3090 - Val Acc: 90.00%
Epoch [5/10] - Train Loss: 0.9192 - Val Loss: 0.3183 - Val Acc: 89.48%
Epoch [6/10] - Train Loss: 0.8444 - Val Loss: 0.2771 - Val Acc: 90.62%
Epoch [7/10] - Train Loss: 0.7510 - Val Loss: 0.2903 - Val Acc: 90.24%
Epoch [8/10] - Train Loss: 0.7066 - Val Loss: 0.3225 - Val Acc: 89.80%
Epoch [9/10] - Train Loss: 0.6691 - Val Loss: 0.3071 - Val Acc: 89.84%
Epoch [10/10] - Train Loss: 0.6279 - Val Loss: 0.2858 - Val Acc: 90.26%
Best Val Acc: 90.62%

Temps d'entraînement Student (pre-trained, scores and features) : 1924.07 


### Strategy 2: Knowledge distillation using a no pre-trained Student:

5. Train the student network (not pre-trained on ImageNet) from scratch on the CIFAR10 dataset
while evaluating it. Ensure proper data split for training, validation, and testing



In [26]:
#student network training
model_student=models.resnet18(pretrained=False)
best_student5_path= os.path.join(save_dir, 'best_student_Q5.pth')
model_student.fc = torch.nn.Linear(model_student.fc.in_features, 10)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model_student.to(device)
best_val_acc = 0.0
#training
start_time = time.time()
for epoch in range(NUM_EPOCHS):
    #train
    model_student.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_student(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    #val
    model_student.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_student(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total
    end_time = time.time()
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(model_student.state_dict(), best_student5_path)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Train Loss: {epoch_loss:.4f} - Val Loss: {epoch_val_loss:.4f} - Val Acc: {val_accuracy:.2f}%")

temps_entrainement = end_time - start_time
print(f"Training time: {temps_entrainement:.2f} seconds")

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Using device: cuda
Epoch [1/10] - Train Loss: 2.3663 - Val Loss: 2.3566 - Val Acc: 11.04%
Epoch [2/10] - Train Loss: 2.3657 - Val Loss: 2.3536 - Val Acc: 11.04%
Epoch [3/10] - Train Loss: 2.3664 - Val Loss: 2.3554 - Val Acc: 10.78%
Epoch [4/10] - Train Loss: 2.3660 - Val Loss: 2.3578 - Val Acc: 10.76%
Epoch [5/10] - Train Loss: 2.3657 - Val Loss: 2.3576 - Val Acc: 11.26%
Epoch [6/10] - Train Loss: 2.3660 - Val Loss: 2.3561 - Val Acc: 10.86%
Epoch [7/10] - Train Loss: 2.3660 - Val Loss: 2.3565 - Val Acc: 11.00%
Epoch [8/10] - Train Loss: 2.3660 - Val Loss: 2.3566 - Val Acc: 10.70%
Epoch [9/10] - Train Loss: 2.3654 - Val Loss: 2.3567 - Val Acc: 11.08%
Epoch [10/10] - Train Loss: 2.3662 - Val Loss: 2.3556 - Val Acc: 10.78%
Training time: 1204.62 seconds


In [27]:
# test
model_student.load_state_dict(torch.load(best_student5_path))
model_student.eval()
test_loss = 0.0
correct = 0
total = 0
start = time.time()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_student(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
end = time.time()
average_test_loss = test_loss / len(test_loader)
test_accuracy = 100 * correct / total

temps_total = end - start
nb_images = len(test_dataset)
fps = nb_images / temps_total
print(f"Teacher Model Test Evaluation:")
print(f"Average Test Loss: {average_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Teacher Model Test Evaluation:
Average Test Loss: 2.3663
Test Accuracy: 11.09%


In [28]:
sample_images, _ = next(iter(test_loader))
dummy_input = sample_images[:1].to(device)

model_student.eval()
with torch.no_grad():
    with FlopCounterMode(display=False) as flop_counter:
        model_student(dummy_input)

flops = flop_counter.get_total_flops()
macs = flops / 2

print(f"FLOPs: {flops / 1e9:.3f} GFLOPs")
print(f"MACs:  {macs / 1e9:.3f} GMACs")

FLOPs: 3.627 GFLOPs
MACs:  1.814 GMACs


6. Freeze the teacher network's parameters and train the student network under the supervision of
the teacher using knowledge distillation at the scores level

In [30]:
studentQ6, hist6, time6 = train_student(save_path=os.path.join(save_dir, "student_Q6.pth"),pretrained=False, use_features=False,logits_metric="mse",feat_metric="mse")
print(f"Temps d'entraînement Student (not pre-trained, scores only) : {time6:.2f} ")


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/10] - Train Loss: 9.1905 - Val Loss: 1.6884 - Val Acc: 58.48%
  -> Saved new best student (58.48%) to /content/drive/MyDrive/saved_models/student_Q6.pth
Epoch [2/10] - Train Loss: 4.2428 - Val Loss: 1.1081 - Val Acc: 72.58%
  -> Saved new best student (72.58%) to /content/drive/MyDrive/saved_models/student_Q6.pth
Epoch [3/10] - Train Loss: 2.7905 - Val Loss: 0.6126 - Val Acc: 80.90%
  -> Saved new best student (80.90%) to /content/drive/MyDrive/saved_models/student_Q6.pth
Epoch [4/10] - Train Loss: 2.1408 - Val Loss: 0.6916 - Val Acc: 80.94%
  -> Saved new best student (80.94%) to /content/drive/MyDrive/saved_models/student_Q6.pth
Epoch [5/10] - Train Loss: 1.6901 - Val Loss: 0.6137 - Val Acc: 82.60%
  -> Saved new best student (82.60%) to /content/drive/MyDrive/saved_models/student_Q6.pth
Epoch [6/10] - Train Loss: 1.4699 - Val Loss: 0.4811 - Val Acc: 85.08%
  -> Saved new best student (85.08%) to /content/drive/MyDrive/saved_models/student_Q6.pth
Epoch [7/10] - Train Loss: 1

7. Freeze the teacher network's parameters and train the student network under the supervision of
the teacher using knowledge distillation at both the scores and feature map levels

In [31]:
studentQ7, hist7, time7 = train_student(save_path=os.path.join(save_dir, "student_Q7.pth"),pretrained=False, use_features=True,logits_metric="mse",feat_metric="mse")
print(f"Temps d'entraînement Student (not pre-trained, scores and features) : {time7:.2f} ")

Epoch [1/10] - Train Loss: 8.9339 - Val Loss: 1.6771 - Val Acc: 61.88%
  -> Saved new best student (61.88%) to /content/drive/MyDrive/saved_models/student_Q7.pth
Epoch [2/10] - Train Loss: 4.0578 - Val Loss: 1.1610 - Val Acc: 70.64%
  -> Saved new best student (70.64%) to /content/drive/MyDrive/saved_models/student_Q7.pth
Epoch [3/10] - Train Loss: 2.7585 - Val Loss: 0.6365 - Val Acc: 80.92%
  -> Saved new best student (80.92%) to /content/drive/MyDrive/saved_models/student_Q7.pth
Epoch [4/10] - Train Loss: 2.1089 - Val Loss: 0.6335 - Val Acc: 81.40%
  -> Saved new best student (81.40%) to /content/drive/MyDrive/saved_models/student_Q7.pth
Epoch [5/10] - Train Loss: 1.7409 - Val Loss: 0.5095 - Val Acc: 85.38%
  -> Saved new best student (85.38%) to /content/drive/MyDrive/saved_models/student_Q7.pth
Epoch [6/10] - Train Loss: 1.4292 - Val Loss: 0.5298 - Val Acc: 84.06%
Epoch [7/10] - Train Loss: 1.2609 - Val Loss: 0.5115 - Val Acc: 85.32%
Epoch [8/10] - Train Loss: 1.0837 - Val Loss: 0.

### Comparisons
8. Compare Strategy 1 and Strategy 2


In [32]:
import pandas as pd
def evaluate_model(model, test_loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()

    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_loss += criterion(outputs, labels).item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = test_loss / len(test_loader)
    accuracy = 100.0 * correct / total

    return {
        "Test Loss": round(avg_loss, 4),
        "Test Acc (%)": round(accuracy, 2),
    }

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_to_test = {
    "StudentQ3 (Pretrained, scores MSE)": studentQ3,
    "StudentQ4 (Pretrained, feat+scores MSE)": studentQ4,
    "StudentQ6 (Not pretrained, scores MSE)": studentQ6,
    "StudentQ7 (Not pretrained, feat+scores MSE)": studentQ7
}

results = {}
for name, model in models_to_test.items():
    results[name] = evaluate_model(model, test_loader, device)

df_results = pd.DataFrame(results).T
display(df_results)

,Test Loss,Test Acc (%)
"StudentQ3 (Pretrained, scores MSE)",0.3391,89.40
"StudentQ4 (Pretrained, feat+scores MSE)",0.3218,89.99
"StudentQ6 (Not pretrained, scores MSE)",0.4470,86.87
"StudentQ7 (Not pretrained, feat+scores MSE)",0.4163,86.99


9. Compare different distance metrics between the teacher and the student (other than MSE)

KL distance

In [34]:
studentQ8, hist8, time8 = train_student(save_path=os.path.join(save_dir, "student_Q8.pth"),pretrained=True, use_features=True,logits_metric="kl",feat_metric="kl")
print(f"Temps d'entraînement Student (pre-trained, scores+features) : {time8:.2f} ")
studentQ9, hist9, time9 = train_student(save_path=os.path.join(save_dir, "student_Q9.pth"),pretrained=False, use_features=True,logits_metric="kl",feat_metric="kl")
print(f"Temps d'entraînement Student (not pre-trained, scores+features) : {time9:.2f} ")
studentQ10, hist10, time10 = train_student(save_path=os.path.join(save_dir, "student_Q10.pth"),pretrained=True, use_features=False,logits_metric="kl",feat_metric="kl")
print(f"Temps d'entraînement Student (pre-trained, scores only) : {time10:.2f} ")
studentQ11, hist11, time11 = train_student(save_path=os.path.join(save_dir, "student_Q11.pth"),pretrained=False, use_features=False,logits_metric="kl",feat_metric="kl")
print(f"Temps d'entraînement Student (not pre-trained, scores only) : {time11:.2f} ")
models_to_test = {
    "Student Pretrained, scores only)": studentQ10,
    "Student Pretrained, feat+scores)": studentQ8,
    "Student Not pretrained, scores only)": studentQ11,
    "Student Not pretrained, feat+scores )": studentQ9
}

results = {}
for name, model in models_to_test.items():
    print("Results using KL metric \n")
    results[name] = evaluate_model(model, test_loader, device)

df_results = pd.DataFrame(results).T
display(df_results)


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/10] - Train Loss: 2.4448 - Val Loss: 0.3903 - Val Acc: 87.28%
  -> Saved new best student (87.28%) to /content/drive/MyDrive/saved_models/student_Q8.pth
Epoch [2/10] - Train Loss: 1.2606 - Val Loss: 0.2948 - Val Acc: 90.50%
  -> Saved new best student (90.50%) to /content/drive/MyDrive/saved_models/student_Q8.pth
Epoch [3/10] - Train Loss: 0.9803 - Val Loss: 0.3053 - Val Acc: 89.50%
Epoch [4/10] - Train Loss: 0.8448 - Val Loss: 0.2810 - Val Acc: 90.66%
  -> Saved new best student (90.66%) to /content/drive/MyDrive/saved_models/student_Q8.pth
Epoch [5/10] - Train Loss: 0.7809 - Val Loss: 0.3194 - Val Acc: 89.50%
Epoch [6/10] - Train Loss: 0.7317 - Val Loss: 0.2904 - Val Acc: 90.42%
Epoch [7/10] - Train Loss: 0.6772 - Val Loss: 0.2929 - Val Acc: 90.66%
Epoch [8/10] - Train Loss: 0.6364 - Val Loss: 0.2854 - Val Acc: 90.50%
Epoch [9/10] - Train Loss: 0.6054 - Val Loss: 0.3573 - Val Acc: 88.54%
Epoch [10/10] - Train Loss: 0.5858 - Val Loss: 0.2745 - Val Acc: 90.86%
  -> Saved new b

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/10] - Train Loss: 7.0249 - Val Loss: 1.3685 - Val Acc: 63.84%
  -> Saved new best student (63.84%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [2/10] - Train Loss: 3.2242 - Val Loss: 0.7064 - Val Acc: 79.40%
  -> Saved new best student (79.40%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [3/10] - Train Loss: 2.1618 - Val Loss: 0.6085 - Val Acc: 81.36%
  -> Saved new best student (81.36%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [4/10] - Train Loss: 1.6507 - Val Loss: 0.4726 - Val Acc: 84.76%
  -> Saved new best student (84.76%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [5/10] - Train Loss: 1.3282 - Val Loss: 0.4813 - Val Acc: 85.20%
  -> Saved new best student (85.20%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [6/10] - Train Loss: 1.1233 - Val Loss: 0.4314 - Val Acc: 86.36%
  -> Saved new best student (86.36%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [7/10] - Train Loss: 0

,Test Loss,Test Acc (%)
"Student Pretrained, scores only)",0.2734,91.38
"Student Pretrained, feat+scores)",0.2957,90.52
"Student Not pretrained, scores only)",0.4895,86.05
"Student Not pretrained, feat+scores )",0.3809,88.09


Cosine Distance

In [35]:
studentQ8, hist8, time8 = train_student(save_path=os.path.join(save_dir, "student_Q8.pth"),pretrained=True, use_features=True,logits_metric="kl",feat_metric="kl")
print(f"Temps d'entraînement Student (pre-trained, scores+features) : {time8:.2f} ")
studentQ9, hist9, time9 = train_student(save_path=os.path.join(save_dir, "student_Q9.pth"),pretrained=False, use_features=True,logits_metric="kl",feat_metric="kl")
print(f"Temps d'entraînement Student (not pre-trained, scores+features) : {time9:.2f} ")
studentQ10, hist10, time10 = train_student(save_path=os.path.join(save_dir, "student_Q10.pth"),pretrained=True, use_features=False,logits_metric="kl",feat_metric="kl")
print(f"Temps d'entraînement Student (pre-trained, scores only) : {time10:.2f} ")
studentQ11, hist11, time11 = train_student(save_path=os.path.join(save_dir, "student_Q11.pth"),pretrained=False, use_features=False,logits_metric="kl",feat_metric="kl")
print(f"Temps d'entraînement Student (not pre-trained, scores only) : {time11:.2f} ")

models_to_test = {
    "Student Pretrained, scores only)": studentQ10,
    "Student Pretrained, feat+scores)": studentQ8,
    "Student Not pretrained, scores only)": studentQ11,
    "Student Not pretrained, feat+scores )": studentQ9
}

results = {}
for name, model in models_to_test.items():
    print("Results using COSINE metric \n")
    results[name] = evaluate_model(model, test_loader, device)

df_results = pd.DataFrame(results).T
display(df_results)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch [1/10] - Train Loss: 2.4591 - Val Loss: 0.3485 - Val Acc: 88.24%
  -> Saved new best student (88.24%) to /content/drive/MyDrive/saved_models/student_Q8.pth
Epoch [2/10] - Train Loss: 1.2554 - Val Loss: 0.3603 - Val Acc: 88.84%
  -> Saved new best student (88.84%) to /content/drive/MyDrive/saved_models/student_Q8.pth
Epoch [3/10] - Train Loss: 0.9799 - Val Loss: 0.3377 - Val Acc: 89.08%
  -> Saved new best student (89.08%) to /content/drive/MyDrive/saved_models/student_Q8.pth
Epoch [4/10] - Train Loss: 0.8550 - Val Loss: 0.3807 - Val Acc: 87.84%
Epoch [5/10] - Train Loss: 0.7836 - Val Loss: 0.2956 - Val Acc: 89.94%
  -> Saved new best student (89.94%) to /content/drive/MyDrive/saved_models/student_Q8.pth
Epoch [6/10] - Train Loss: 0.7154 - Val Loss: 0.2895 - Val Acc: 90.62%
  -> Saved new best student (90.62%) to /content/drive/MyDrive/saved_models/student_Q8.pth
Epoch [7/10] - Train Loss: 0.6731 - Val Loss: 0.2719 - Val Acc: 90.80%
  -> Saved new best student (90.80%) to /content

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch [1/10] - Train Loss: 6.6309 - Val Loss: 1.4354 - Val Acc: 63.26%
  -> Saved new best student (63.26%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [2/10] - Train Loss: 3.0738 - Val Loss: 0.8965 - Val Acc: 75.20%
  -> Saved new best student (75.20%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [3/10] - Train Loss: 2.0546 - Val Loss: 0.5353 - Val Acc: 82.76%
  -> Saved new best student (82.76%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [4/10] - Train Loss: 1.5904 - Val Loss: 0.5712 - Val Acc: 82.54%
Epoch [5/10] - Train Loss: 1.3238 - Val Loss: 0.4320 - Val Acc: 86.26%
  -> Saved new best student (86.26%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [6/10] - Train Loss: 1.0973 - Val Loss: 0.4327 - Val Acc: 86.42%
  -> Saved new best student (86.42%) to /content/drive/MyDrive/saved_models/student_Q9.pth
Epoch [7/10] - Train Loss: 0.9511 - Val Loss: 0.3657 - Val Acc: 88.06%
  -> Saved new best student (88.06%) to /content

,Test Loss,Test Acc (%)
"Student Pretrained, scores only)",0.2639,91.50
"Student Pretrained, feat+scores)",0.2999,90.37
"Student Not pretrained, scores only)",0.4107,87.20
"Student Not pretrained, feat+scores )",0.3667,88.21
